# Signal Loss: Filtered vs Unfiltered Power Spectra

Compares outputs of `power_spectrum_summary.ipynb` run once on the **unfiltered** merge and once on the **filtered** merge:

$$\text{loss} = 1 - \frac{|\Delta^2_{\rm filt}|}{|\Delta^2_{\rm unfilt}|}$$

* **Section 1 – Spherical (1D):** `{CASENAME}.deltasq.1d.spw{NN}.npz`. The summary notebook sets the noise to infinity inside the wedge + buffer before spherical averaging, so these statistics are *beyond-the-wedge only*.
* **Section 2 – Cylindrical (2D):** `{CASENAME}.deltasq.2d.spw{NN}.npz`. All delays are kept, so loss is reported separately *inside* and *outside* the wedge.

Both runs must use the same bands, k-bins, and baseline set.

## IMPORTS

In [ ]:
import numpy as np
import hera_pspec as hp
import pandas as pd
import csv
import toml
import matplotlib.pyplot as plt
import os
from pathlib import Path

## PARAMETERS

In [ ]:
# ---- Parameters (set via papermill if you like) ----
UNFILT_CASENAME: str = "unfiltered"
FILT_CASENAME: str = "filtered"
OUT_DIR: str = "."
BANDS: str = "1,2,3,5,6,9,10,13"   # 1-indexed, same as BANDS_TO_USE in the summary notebook
WEDGE_BUFFER_NS: float = 500.0     # must match the summary notebook
MIN_SNR: float = 0.0               # include a pixel/bin only if |Delta^2_unfilt| > MIN_SNR * 1sigma
MAX_ERR: float = 1e20              # error bars above this count as unconstrained

BAND_LIST = [int(b) for b in BANDS.split(",")]
OUT_DIR = Path(OUT_DIR)

In [ ]:
UNFILT_DIR: str = ""
FILT_DIR: str = ""
PSPEC_OUT_DIR: str = ""            # run_batch output: .../pspecs_and_horizons/pspec_out (has filtered/ and unfiltered/)
UNFILT_MERGED_PSPEC: str = ""      # optional: merged unfiltered .pspec.h5, to cross-check its baselines
FILT_MERGED_PSPEC: str = ""        # optional: merged filtered .pspec.h5

## TOML CONFIGS

In [ ]:
#------------LOADING TOML CONFIGS------------#

TOML_FILE = os.environ.get('TOML_FILE',
        '/home/Kwuzard/Projects/HERA_HONS/src/HONS/h6c_pspec_11band.toml')

# Load configuration from the toml and inject into globals.
toml_options = toml.load(TOML_FILE)
for toml_section in ['GLOBAL_OPTS', 'POSTPROCESS_AND_PSPEC_OPTS']:
    if toml_section not in toml_options:
        continue
    print(f'\nLoading config from [{toml_section}] in {TOML_FILE}:')
    for key, val in toml_options[toml_section].items():
        globals()[key.upper()] = val
        print(f'  {key.upper()} = {val!r}')

### CONSTANTS


In [ ]:
C = 299792458.0  # m/s

## Section 0: Baseline accounting

Reads the per-baseline `is_used.csv` written by `run_batch.ipynb` and checks which `BLPAIR_*` folders actually have both a filtered and an unfiltered power spectrum. This notebook only sees the merged, averaged outputs, so it **cannot drop baselines itself**: the merge step must use only the baselines marked *used* (the same set for both merges). This section reports who was dropped and, optionally, whether the merged files match.

In [ ]:
def read_usage(path):
    # (is_used, reason) from is_used.csv; (None, "") if the file is missing. Old files without a reason row give "".
    flag, reason = None, ""
    try:
        with open(path, newline="", encoding="utf-8") as fh:
            for row in csv.reader(fh):
                if len(row) < 2:
                    continue
                key = row[0].strip()
                if key == "is_used":
                    flag = row[1].strip().lower() == "true"
                elif key == "reason":
                    reason = row[1].strip()
    except FileNotFoundError:
        return None, ""
    return flag, reason

def reason_category(reason):
    # Group the free-text reasons written by run_batch. Order matters: 'unfiltered ...' contains 'filtered ...',
    # and the tolerance message also mentions 'horizon'.
    r = reason.lower()
    if not r:
        return "no reason recorded"
    if "tolerance" in r:
        return "signal loss above tolerance"
    if "eigenvalue cutoff" in r:
        return "eigenvalue cutoff search failed"
    if "determine horizon" in r:
        return "could not determine horizon"
    if "unfiltered power spectrum" in r:
        return "unfiltered pspec not created"
    if "filtered power spectrum" in r:
        return "filtered pspec not created"
    return "other"

def account_baselines(pspec_out_dir):
    root = Path(pspec_out_dir)
    filt, unfilt = root / "filtered", root / "unfiltered"
    names = sorted({p.name for p in filt.glob("BLPAIR_*")} | {p.name for p in unfilt.glob("BLPAIR_*")})
    rows = []
    for name in names:
        has_f = any((filt / name).glob("*.tavg.pspec.h5"))
        has_u = any((unfilt / name).glob("*.tavg.pspec.h5"))
        flag, reason = read_usage(filt / name / "is_used.csv")
        category = ""
        if flag is None:
            status = "unknown: no is_used.csv"
        elif flag is False:
            status, category = "dropped", reason_category(reason)
        elif not has_u or not has_f:
            missing = " and ".join(k for k, h in [("unfiltered", has_u), ("filtered", has_f)] if not h)
            status = f"inconsistent: flagged used but missing {missing} pspec"
        else:
            status = "used"
        rows.append(dict(blpair=name, has_unfiltered=has_u, has_filtered=has_f, is_used_flag=flag,
                         status=status, category=category, reason=reason))
    return pd.DataFrame(rows)

def report_baselines(df):
    print(f"{len(df)} baselines found; {int((df.status == 'used').sum())} used, "
          f"{int((df.status == 'dropped').sum())} dropped.")
    dropped = df[df.status == "dropped"]
    for cat, grp in dropped.groupby("category"):
        print(f"\nDROPPED - {cat.upper()} ({len(grp)}):")
        for _, r in grp.iterrows():
            print(f"  {r.blpair}: {r.reason}")
    for status in [s for s in df.status.unique() if s not in ("used", "dropped")]:
        names = df.loc[df.status == status, "blpair"].tolist()
        print(f"\n{status.upper()} ({len(names)}):")
        print("  " + ", ".join(names))

In [ ]:
if PSPEC_OUT_DIR:
    df_bl = account_baselines(PSPEC_OUT_DIR)
    df_bl.to_csv(OUT_DIR / "baseline_accounting.csv", index=False)
    report_baselines(df_bl)
    used_names = set(df_bl.loc[df_bl.status == "used", "blpair"])
else:
    print("PSPEC_OUT_DIR not set; skipping baseline accounting.")
    df_bl, used_names = None, None

In [ ]:
def pair_from_name(name):
    a, b = name.replace("BLPAIR_", "").split("_")
    conv = lambda t: int(t) if t.lstrip("-").isdigit() else t
    return tuple(sorted((conv(a), conv(b)), key=str))

def ant_pairs_from_blpairs(blpairs):
    # blpairs look like ((a, b), (c, d)); collect every baseline that appears in any blpair
    return {tuple(sorted(tuple(bl), key=str)) for blp in blpairs for bl in blp}

def ant_pairs_in_merge(path):
    uvp = hp.UVPSpec()
    uvp.read_hdf5(path)
    return ant_pairs_from_blpairs(uvp.get_blpairs())

def compare_merge(used_names, df_bl, unfilt_path, filt_path):
    used = {pair_from_name(n) for n in used_names}
    unused = {pair_from_name(n) for n in df_bl.blpair} - used
    in_u, in_f = ant_pairs_in_merge(unfilt_path), ant_pairs_in_merge(filt_path)
    checks = {
        "in unfiltered merge but not in filtered merge": in_u - in_f,
        "in filtered merge but not in unfiltered merge": in_f - in_u,
        "flagged not-used but present in the filtered merge": unused & in_f,
        "flagged not-used but present in the unfiltered merge": unused & in_u,
        "flagged used but missing from the filtered merge": used - in_f,
        "flagged used but missing from the unfiltered merge": used - in_u,
    }
    ok = True
    for label, pairs in checks.items():
        if pairs:
            ok = False
            print(f"WARNING {label} ({len(pairs)}): " + ", ".join(f"{a}_{b}" for a, b in sorted(pairs, key=str)))
    if ok:
        print("Merged files contain exactly the baselines flagged as used, in both runs.")
    return checks

In [ ]:
if UNFILT_MERGED_PSPEC and FILT_MERGED_PSPEC and df_bl is not None:
    merge_checks = compare_merge(used_names, df_bl, UNFILT_MERGED_PSPEC, FILT_MERGED_PSPEC)
else:
    print("Merged pspec paths not set; skipping merge cross-check.")

## Section 1: Spherical statistics (beyond the wedge only)

In [ ]:
def load_1d(folder, casename, band):
    f = np.load(Path(folder) / f"{casename}.deltasq.1d.spw{band:02d}.npz")
    return f["k_h_on_mpc"], f["deltasq_mk2"], np.real(f["onesigma_error"]), float(f["z"])

def good_mask(d_u, e_u, e_f, min_snr, max_err):
    # constrained in BOTH runs, nonzero unfiltered power, optional SNR cut
    good = np.isfinite(e_u) & np.isfinite(e_f) & (e_u < max_err) & (e_f < max_err)
    good &= np.abs(d_u) > 0
    good &= np.abs(d_u) > min_snr * np.where(np.isfinite(e_u), e_u, 0)
    return good

def loss_array(d_f, d_u, good):
    loss = np.full(d_u.shape, np.nan)
    loss[good] = 1 - np.abs(d_f[good]) / np.abs(d_u[good])
    return loss

def basic_stats(v):
    if v.size == 0:
        return dict(n=0, max_loss=np.nan, min_loss=np.nan, mean_loss=np.nan, var_loss=np.nan)
    return dict(n=int(v.size), max_loss=v.max(), min_loss=v.min(), mean_loss=v.mean(),
                var_loss=v.var(ddof=1) if v.size > 1 else np.nan)

In [ ]:
sph_rows, sph_loss = [], {}
for band in BAND_LIST:
    k_u, d_u, e_u, z = load_1d(UNFILT_DIR, UNFILT_CASENAME, band)
    k_f, d_f, e_f, _ = load_1d(FILT_DIR, FILT_CASENAME, band)
    assert np.allclose(k_u, k_f), f"k-bins differ in band {band}; use identical KBINS settings"
    good = good_mask(d_u, e_u, e_f, MIN_SNR, MAX_ERR)
    loss = loss_array(d_f, d_u, good)
    sph_loss[band] = (k_u, loss, z)
    st = basic_stats(loss[good])
    sph_rows.append(dict(band=band, z=z, n_kbins=st["n"], max_loss=st["max_loss"], min_loss=st["min_loss"],
                         mean_loss=st["mean_loss"], var_loss=st["var_loss"],
                         passes=bool(st["n"] and st["max_loss"] <= FIT_REQUIRES_MAX_SIGNAL_LOSS_BEYOND_HORIZON)))
df_sph = pd.DataFrame(sph_rows)
df_sph.to_csv(OUT_DIR / "signal_loss_spherical.csv", index=False)
df_sph

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for band, (k, loss, z) in sph_loss.items():
    ax.plot(k, loss, marker='o', ms=3, label=f"Band {band} (z={z:.1f})")
ax.axhline(FIT_REQUIRES_MAX_SIGNAL_LOSS_BEYOND_HORIZON, color='k', ls='--', lw=1)
ax.set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]"); ax.set_ylabel(r"$1 - |\Delta^2_{filt}| / |\Delta^2_{unfilt}|$")
ax.legend(fontsize=7); plt.tight_layout()

## Section 2: Cylindrical statistics (inside and outside the wedge)

The wedge boundary for a baseline group of length $|b|$ is $\tau_h = |b|/c + \text{WEDGE\_BUFFER\_NS}$, converted to $k_\parallel$ with the same cosmology as the summary notebook. Pixels with $k_\parallel \le k_{\parallel,h}$ are *inside*.

* **Inside the wedge:** integrated loss $= 1 - \sum|\Delta^2_{\rm filt}| / \sum|\Delta^2_{\rm unfilt}|$ over inside pixels (power-weighted, so dominated by the brightest foreground pixels), plus the plain mean per-pixel loss. Here this mostly measures foreground suppression.
* **Outside the wedge:** max, min, mean, and variance of the per-pixel loss, with the location of the maximum.

In [ ]:
def get_cosmo():
    return hp.conversions.Cosmo_Conversions()

cnv = get_cosmo()

def load_2d(folder, casename, band):
    f = np.load(Path(folder) / f"{casename}.deltasq.2d.spw{band:02d}.npz")
    return dict(dsq=f["deltasq"], err=np.real(f["errorbars_onesigma"]),
                kperp=f["kperp_h_on_mpc"], kpara=f["kpara_h_on_mpc"], z=float(f["z"]))

def wedge_kpara(kperp, z, buffer_ns):
    # k_parallel of the wedge boundary for each baseline group (h/Mpc)
    b = kperp / cnv.bl_to_kperp(z)                     # metres
    return cnv.tau_to_kpara(z) * (b / C + buffer_ns * 1e-9)

def match_rows(kperp_u, kperp_f, rtol=1e-6):
    # baseline groups present in both runs (the FRF-loss cut can differ between them)
    iu, jf = [], []
    for i, k in enumerate(kperp_u):
        j = int(np.argmin(np.abs(kperp_f - k)))
        if np.isclose(kperp_f[j], k, rtol=rtol):
            iu.append(i); jf.append(j)
    return np.array(iu, dtype=int), np.array(jf, dtype=int)

def cyl_stats(u, f):
    iu, jf = match_rows(u["kperp"], f["kperp"])
    assert iu.size > 0, "no common baseline groups between the two runs"
    assert np.allclose(u["kpara"], f["kpara"]), "delay grids differ"
    d_u, d_f = u["dsq"][iu], f["dsq"][jf]
    e_u, e_f = u["err"][iu], f["err"][jf]
    kperp, kpara = u["kperp"][iu], u["kpara"]
    inside = kpara[None, :] <= wedge_kpara(kperp, u["z"], WEDGE_BUFFER_NS)[:, None]
    good = good_mask(d_u, e_u, e_f, MIN_SNR, MAX_ERR)
    loss = loss_array(d_f, d_u, good)

    def region(sel_in):
        m = good & (inside if sel_in else ~inside)
        return m

    def summarize(m):
        v = loss[m]
        st = basic_stats(v)
        st["int_loss"] = (1 - np.abs(d_f[m]).sum() / np.abs(d_u[m]).sum()) if m.any() else np.nan
        return st

    # per-baseline-group table
    rows = []
    for i in range(kperp.size):
        row = dict(kperp=kperp[i], kpara_wedge=wedge_kpara(kperp[i:i+1], u["z"], WEDGE_BUFFER_NS)[0])
        for name, sel_in in [("in", True), ("out", False)]:
            m = (good & (inside if sel_in else ~inside))[i:i+1]
            sub = lambda a: a[i:i+1][m]
            v = loss[i:i+1][m]
            st = basic_stats(v)
            row.update({f"n_{name}": st["n"], f"max_loss_{name}": st["max_loss"], f"mean_loss_{name}": st["mean_loss"]})
            row[f"int_loss_{name}"] = (1 - np.abs(sub(d_f)).sum() / np.abs(sub(d_u)).sum()) if st["n"] else np.nan
        rows.append(row)
    per_bl = pd.DataFrame(rows)

    # pooled-over-baselines summary for this band
    s_in, s_out = summarize(region(True)), summarize(region(False))
    out = dict(z=u["z"], n_blgroups=int(kperp.size),
               n_in=s_in["n"], int_loss_in=s_in["int_loss"], mean_loss_in=s_in["mean_loss"],
               max_loss_in=s_in["max_loss"], min_loss_in=s_in["min_loss"],
               n_out=s_out["n"], max_loss_out=s_out["max_loss"], min_loss_out=s_out["min_loss"],
               mean_loss_out=s_out["mean_loss"], var_loss_out=s_out["var_loss"])
    m_out = region(False)
    if m_out.any():
        idx = np.unravel_index(np.nanargmax(np.where(m_out, loss, -np.inf)), loss.shape)
        out.update(max_out_kperp=kperp[idx[0]], max_out_kpara=kpara[idx[1]])
    else:
        out.update(max_out_kperp=np.nan, max_out_kpara=np.nan)
    out["passes_out"] = bool(s_out["n"] and s_out["max_loss"] <= FIT_REQUIRES_MAX_SIGNAL_LOSS_BEYOND_HORIZON)
    return out, per_bl, (kperp, kpara, loss, inside)

In [ ]:
cyl_rows, cyl_per_bl, cyl_maps = [], [], {}
for band in BAND_LIST:
    u = load_2d(UNFILT_DIR, UNFILT_CASENAME, band)
    f = load_2d(FILT_DIR, FILT_CASENAME, band)
    summary, per_bl, maps = cyl_stats(u, f)
    cyl_rows.append(dict(band=band, **summary))
    per_bl.insert(0, "band", band)
    cyl_per_bl.append(per_bl)
    cyl_maps[band] = (maps, summary["z"])

df_cyl = pd.DataFrame(cyl_rows)
df_cyl_bl = pd.concat(cyl_per_bl, ignore_index=True)
df_cyl.to_csv(OUT_DIR / "signal_loss_cylindrical_per_band.csv", index=False)
df_cyl_bl.to_csv(OUT_DIR / "signal_loss_cylindrical_per_baseline.csv", index=False)
df_cyl

In [ ]:
# Per-baseline-group detail (inside: integrated loss; outside: max loss)
df_cyl_bl[["band", "kperp", "kpara_wedge", "n_in", "int_loss_in", "n_out", "max_loss_out", "mean_loss_out"]]

In [ ]:
nb_ = len(BAND_LIST)
fig, axes = plt.subplots(int(np.ceil(nb_ / 2)), 2, figsize=(12, 3.2 * int(np.ceil(nb_ / 2))), squeeze=False)
for ax, (band, ((kperp, kpara, loss, inside), z)) in zip(axes.ravel(), cyl_maps.items()):
    im = ax.pcolormesh(kperp, kpara, loss.T, shading='nearest', cmap='viridis', vmin=-0.1, vmax=1)
    kp = np.linspace(kperp.min(), kperp.max(), 100)
    ax.plot(kp, wedge_kpara(kp, z, WEDGE_BUFFER_NS), 'r--', lw=1)
    ax.set_ylim(kpara.min(), kpara.max())
    ax.set_title(f"Band {band} (z={z:.1f})", fontsize=9)
    ax.set_xlabel(r"$k_\perp$ [$h$/Mpc]"); ax.set_ylabel(r"$k_\parallel$ [$h$/Mpc]")
fig.colorbar(im, ax=axes, label="signal loss"); 